# Speech → ASL gloss demo (local Jupyter)

**What this notebook does.** Demo, TODO §13 Phase 4: the team's Colab main loop (record →
Whisper → gloss → display) running locally, through `sb.synthesize.pipeline`. No Colab
APIs, no Drive, no signasl.org scraping.

- **§1 live:** record from the microphone, then transcribe and gloss. Skipped, with a
  message, on a machine with no input device (a remote PC).
- **§2 from a file:** any audio file.
- **§2b no microphone:** type a sentence, a Windows SAPI voice speaks it, and the full
  pipeline (ASR → gloss) runs on that audio.
- **§3 typed text:** gloss only (no ASR), to try the engines quickly.

The gloss is shown as chips: **green** = GISLR (our recognizer knows the sign), **blue** =
WLASL only, **orange** = no sign (fingerspell). If WLASL videos are under
`data/raw/wlasl/videos/`, the trimmed clips play under the gloss. Rendering is otherwise
deferred (TODO §13).

No artifacts: outputs are shown, not saved. The engine and Whisper model come from
`configs/speech.pipeline.json`; override them in the setup cell.

## Setup

In [1]:
# ============================================================
# Imports, configs, load Whisper + the gloss engine
# ============================================================
import json
from pathlib import Path

from IPython.display import HTML, display

from sb.core.paths import EXPERIMENTS_DIR
from sb.synthesize import asr, display as D, lexicon
from sb.synthesize.gloss import GlossEngine
from sb.synthesize.pipeline import SpeechToGloss

ENGINE = None       # None -> config "engine"; or rules_v1 / rules_v2 / hybrid_team / hybrid_guarded
ASR_MODEL = None    # None -> config asr.model; e.g. "large-v3-turbo"

CONFIG_DIR = EXPERIMENTS_DIR / "synthesis" / "configs"
SPEECH = json.loads((CONFIG_DIR / "speech.pipeline.json").read_text(encoding="utf-8"))
T2G = json.loads((CONFIG_DIR / "aslg.text2gloss.json").read_text(encoding="utf-8"))
NB_DIR = EXPERIMENTS_DIR / "synthesis"

PIPE = SpeechToGloss.from_configs(SPEECH, T2G, engine=ENGINE, asr_model=ASR_MODEL)
LEX = lexicon.Lexicon()
MIC = asr.microphone()
print(f"microphone: {MIC or 'none -> §1 is skipped; use §2 or §2b'}")
print(f"ASR: {PIPE.asr.model_name} on {PIPE.asr.device} ({PIPE.asr.compute_type})")
print(f"gloss engine: {PIPE.engine.name} (T5 {'loaded' if PIPE.engine.uses_t5 else 'not available -> rules'})")


def show(res):
    print(f"English : {res.english or '(no speech detected)'}")
    print(f"Gloss   : {res.gloss.gloss}")
    print(f"ASR {res.asr_seconds * 1000:.0f} ms for {res.audio_seconds:.1f} s of audio · gloss {res.gloss_seconds * 1000:.0f} ms")
    for s in res.gloss.per_sentence:
        if s.get("reasons") and s["reasons"] != ["no_t5"]:  # no checkpoint: nothing was rejected
            print(f"   guard rejected T5 '{s.get('t5')}' ({', '.join(s['reasons'])}) -> {s['source']}")
    if res.gloss.gloss:
        display(HTML(D.lexicon_view(LEX, res.gloss.gloss, NB_DIR)))

microphone: none -> §1 is skipped; use §2 or §2b
ASR: large-v3 on cuda (float16)
gloss engine: hybrid_guarded (T5 not available -> rules)


## 1. Live: record and translate

In [2]:
# ============================================================
# Record SECONDS from the default mic -> English -> gloss
# ============================================================
SECONDS = SPEECH["recording"]["seconds"]

if MIC is None:
    print("no microphone on this machine (remote PC?) -- skipped. Use §2 (audio file) or §2b (typed text -> speech).")
else:
    print(f"recording {SECONDS} s from {MIC} ... speak now")
    audio = asr.record(SECONDS)
    show(PIPE(audio))

no microphone on this machine (remote PC?) -- skipped. Use §2 (audio file) or §2b (typed text -> speech).


## 2. From an audio file

In [3]:
# ============================================================
# Audio file -> English -> gloss
# ============================================================
AUDIO_FILE = None  # e.g. Path("data/raw/speech30/01.wav") (relative to the repo root)

if AUDIO_FILE is None:
    print("set AUDIO_FILE")
else:
    from sb.core.paths import ROOT_DIR

    show(PIPE(ROOT_DIR / AUDIO_FILE))

set AUDIO_FILE


## 2b. No microphone: typed text → synthesized speech → the full pipeline

A Windows SAPI voice speaks `TEXT` into `data/temp/` (throwaway). You can play it below,
and the same audio goes through Whisper and the gloss engine. Synthetic speech is
clean, so this shows the pipeline working, not real-user accuracy.

In [4]:
# ============================================================
# Typed text -> SAPI speech -> Whisper -> gloss
# ============================================================
from IPython.display import Audio

from sb.core.paths import TEMP_DIR
from sb.synthesize import tts

TEXT = "Yesterday my brother and I went to the store. Where is the cat?"
VOICE = None  # None -> first installed voice; e.g. "Microsoft Zira Desktop"

voices = tts.sapi_voices()
if not voices:
    print("no SAPI voices (not Windows?) -- use §2 with an audio file")
else:
    wav = TEMP_DIR / "demo_tts.wav"
    info = tts.sapi_speak(TEXT, wav, VOICE or voices[0])
    print(f"spoken by {VOICE or voices[0]}: {info['audio_seconds']:.1f} s of audio in {info['seconds']:.2f} s")
    display(Audio(str(wav)))
    show(PIPE(wav))

spoken by Microsoft David Desktop: 4.9 s of audio in 0.22 s


English : Yesterday my brother and I went to the store. Where is the cat?
Gloss   : YESTERDAY MY BROTHER AND ME GO STORE | CAT WHERE
ASR 502 ms for 4.9 s of audio · gloss 817 ms


## 3. Typed text (no ASR): compare the engines

In [5]:
# ============================================================
# English text -> gloss, every engine side by side
# ============================================================
TEXT = "If it rains tomorrow, I will stay home. Can you help me?"

for name in T2G["eval"]["engines"]:
    g = GlossEngine(name, T2G["t5"])(TEXT)
    print(f"{name:15s} {g.gloss}")
display(HTML(D.lexicon_view(LEX, GlossEngine(PIPE.engine.name, T2G["t5"])(TEXT).gloss, NB_DIR)))

rules_v1        IT RAIN TOMORROW I FUTURE STAY HOME | YOU HELP I
rules_v2        TOMORROW IF RAIN ME STAY HOME | YOU HELP ME CAN
hybrid_team     IT RAIN TOMORROW I FUTURE STAY HOME | YOU HELP I
hybrid_guarded  TOMORROW IF RAIN ME STAY HOME | YOU HELP ME CAN
